
# Demo Masked Language Models (MLM)

Notebook này được chuẩn hóa lại thành một bài hoàn chỉnh chỉ tập trung vào:
- Khái niệm Masked Language Models
- Cách hoạt động của MLM
- Demo dự đoán token bị mask
- Thực hành với Transformer/HuggingFace

Notebook được tối giản để chạy demo trực tiếp.


# Demo Chương 9: Masked Language Models (BERT)
1. **Contextual Embeddings**: Khám phá cách BERT tạo ra các vector khác nhau cho cùng một từ tùy thuộc vào ngữ cảnh.
2. **Fine-tuning**: Tinh chỉnh BERT cho bài toán Phân loại văn bản (Sequence Classification) sử dụng tập dữ liệu IMDB Movie Reviews.

## Phần 1: Contextual Embeddings và Word Sense

In [1]:
!pip install -q transformers datasets evaluate torch scikit-learn

import torch
from transformers import BertTokenizer, BertModel
from torch.nn.functional import cosine_similarity

# Tải tokenizer và pre-trained model của BERT (phiên bản base, uncased)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased', output_hidden_states=True)
model.eval()

# 3 câu ví dụ với từ "mouse" mang 2 nghĩa khác nhau:
# Câu 1: Nghĩa động vật (con chuột)
# Câu 2: Nghĩa động vật (con chuột)
# Câu 3: Nghĩa thiết bị máy tính (chuột máy tính)
sentences =[
    "A small quiet animal like a mouse.",
    "The cat chased the mouse across the kitchen.",
    "I bought a new wireless mouse for my computer."
]

def get_word_embedding(sentence, target_word):
    # Tokenize câu và lấy index của từ target
    inputs = tokenizer(sentence, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    word_index = tokens.index(target_word)

    # Chạy qua BERT
    with torch.no_grad():
        outputs = model(**inputs)

    # Lấy hidden states từ layer cuối cùng (layer 12)
    last_hidden_state = outputs.hidden_states[-1]

    # Trả về vector của từ mong muốn
    return last_hidden_state[0, word_index, :]

# Lấy embedding của từ "mouse" trong 3 câu
emb_animal_1 = get_word_embedding(sentences[0], "mouse")
emb_animal_2 = get_word_embedding(sentences[1], "mouse")
emb_computer = get_word_embedding(sentences[2], "mouse")

# So sánh độ tương đồng (Cosine Similarity)
sim_animal_animal = cosine_similarity(emb_animal_1.unsqueeze(0), emb_animal_2.unsqueeze(0)).item()
sim_animal_computer = cosine_similarity(emb_animal_1.unsqueeze(0), emb_computer.unsqueeze(0)).item()

print(f"Similarity giữa 'mouse' (động vật) và 'mouse' (động vật): {sim_animal_animal:.4f}")
print(f"Similarity giữa 'mouse' (động vật) và 'mouse' (máy tính): {sim_animal_computer:.4f}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Similarity giữa 'mouse' (động vật) và 'mouse' (động vật): 0.7389
Similarity giữa 'mouse' (động vật) và 'mouse' (máy tính): 0.6258


## Phần 2: Fine-Tuning for Sequence Classification

In [2]:
from datasets import load_dataset

# Load dataset IMDB (tương đương với bộ trên Kaggle)
dataset = load_dataset("imdb")

# Lấy một sample nhỏ để demo chạy nhanh hơn
small_train_dataset = dataset["train"].shuffle(seed=42).select(range(1000))
small_test_dataset = dataset["test"].shuffle(seed=42).select(range(200))

print("Ví dụ 1 review:", small_train_dataset[0]['text'][:200], "...\nLabel:", small_train_dataset[0]['label'])

# Hàm Tokenize dữ liệu sử dụng WordPiece (như mô tả trong sách)
def tokenize_function(examples):
    # Padding và Truncation để đưa về cùng kích thước (max 512 tokens)
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# Map hàm tokenize vào dataset
tokenized_train = small_train_dataset.map(tokenize_function, batched=True)
tokenized_test = small_test_dataset.map(tokenize_function, batched=True)

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Ví dụ 1 review: There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. F ...
Label: 1


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

### Khởi tạo Mô hình cho Sequence Classification
HuggingFace cung cấp sẵn class `AutoModelForSequenceClassification`. Class này tự động lấy pre-trained BERT và thêm một mạng Linear Classifier (Classifier Head) đè lên output của token `[CLS]` (đúng như công thức $y = \text{softmax}(\mathbf{h}_{CLS}^L \mathbf{W}_C)$).

In [3]:
import numpy as np
import evaluate
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Load model với Head gồm 2 class (Positive/Negative)
model_cls = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# Khởi tạo hàm tính độ đo (Accuracy)
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Thiết lập tham số training (Training Regimes - Mục 9.2.3)
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5, # Learning rate nhỏ đặc trưng của quá trình Fine-tuning
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3, # Thường fine-tune BERT chỉ cần 2-4 epochs
    eval_strategy="epoch", # ĐÃ SỬA: dùng eval_strategy thay vì evaluation_strategy
    logging_dir='./logs',
)

# Khởi tạo Trainer
trainer = Trainer(
    model=model_cls,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

# BẮT ĐẦU FINE-TUNING!
# Quá trình này sẽ điều chỉnh nhẹ trọng số của BERT (backpropagation qua toàn bộ mạng)
print("Bắt đầu quá trình Fine-tuning...")
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Bắt đầu quá trình Fine-tuning...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.339191,0.875000
2,No log,0.382232,0.860000
3,No log,0.418103,0.895000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=375, training_loss=0.2752585652669271, metrics={'train_runtime': 18393.661, 'train_samples_per_second': 0.163, 'train_steps_per_second': 0.02, 'total_flos': 789333166080000.0, 'train_loss': 0.2752585652669271, 'epoch': 3.0})

### Kiểm tra mô hình sau khi Fine-Tuning

In [4]:
# Test với câu văn tự định nghĩa
test_reviews =[
    "This movie was absolutely fantastic! The acting was great and the plot was thrilling.",
    "What a waste of time. The directing was terrible and the story made no sense.",
    "It was okay, not the best but I had a good time watching it."
]

# Đưa model về chế độ dự đoán
model_cls.eval()

for review in test_reviews:
    # Bước 1: Thêm token [CLS], [SEP] và tạo WordPiece tokens
    inputs = tokenizer(review, return_tensors="pt", truncation=True, padding=True)

    # Bước 2: Chuyển dữ liệu lên cùng device với model (GPU nếu có)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_cls.to(device)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Bước 3: Pass qua model và lấy Logits
    with torch.no_grad():
        outputs = model_cls(**inputs)
        logits = outputs.logits

    # Bước 4: Dùng Softmax / Argmax để ra class cuối cùng (0: Tiêu cực, 1: Tích cực)
    predicted_class = torch.argmax(logits, dim=1).item()
    confidence = torch.softmax(logits, dim=1)[0][predicted_class].item()

    label_map = {0: "Negative (Tiêu cực)", 1: "Positive (Tích cực)"}
    print(f"Review: '{review}'")
    print(f"-> Dự đoán: {label_map[predicted_class]} (Độ tự tin: {confidence*100:.2f}%)\n")

Review: 'This movie was absolutely fantastic! The acting was great and the plot was thrilling.'
-> Dự đoán: Positive (Tích cực) (Độ tự tin: 99.58%)

Review: 'What a waste of time. The directing was terrible and the story made no sense.'
-> Dự đoán: Negative (Tiêu cực) (Độ tự tin: 99.43%)

Review: 'It was okay, not the best but I had a good time watching it.'
-> Dự đoán: Positive (Tích cực) (Độ tự tin: 98.72%)

